In [ ]:
import pandas as pd 
import numpy as np
import pickle

In [2]:
df = pd.read_csv('./data/combined.csv')
df.head(1)

,ticker,company_name,evidence_text,form_type,filing_date,source_url,company name
0,TSLA,"Tesla, Inc.",The fatal Dec. 29 crash of aTeslavehicle in So...,news_article,NaN,https://www.cnbc.com/2019/12/31/us-auto-safety...,NaN


In [3]:
# Filter tickers with ≥2 evidence
group_sizes = df.groupby("ticker").size()
valid_tickers = group_sizes[group_sizes >= 2].index

df_filtered = df[df["ticker"].isin(valid_tickers)].copy()

# Deterministic sampling (3-5 per group)
rng = np.random.default_rng(210) # random seed for reproducibility 

def sample_group(group):
    n = len(group)

    # determine sample size deterministically
    if n <= 3:
        k = n
    else:
        k = rng.integers(3, min(5, n) + 1)

    return group.sample(n=k, random_state=210)

In [4]:
# apply sampling
df_sampled = (
    df_filtered
    .groupby("ticker", group_keys=True)
    .apply(sample_group)
    .reset_index(level=0) # bring back ticker as column 
    .reset_index(drop=True)
)

# debugging columns 
# original group size (before sampling)
df_sampled["original_group_size"] = df_filtered["ticker"].map(group_sizes)

# sampled group size (after sampling)
df_sampled["sampled_group_size"] = df_sampled.groupby("ticker")["ticker"].transform("size")
# sanity checks 

# Ensure all retained tickers had ≥2 evidence originally
assert (group_sizes[valid_tickers] >= 2).all()

# Ensure sampled groups are within expected bounds
sampled_sizes = df_sampled.groupby("ticker").size()
assert (sampled_sizes >= 2).all()
assert (sampled_sizes <= 5).all()

df_sampled.head(5)

,ticker,company_name,evidence_text,form_type,filing_date,source_url,company name,original_group_size,sampled_group_size
0,AAL,American Airlines Group Inc.,"• Trump inauguration, Q4 earnings season will ...",news_article,NaN,https://finance.yahoo.com/news/1-stock-buy-1-s...,NaN,444.0,2
1,AAL,American Airlines Group Inc.,Robinhood CEO Vlad Tenev explained what he cal...,news_article,NaN,https://www.cnn.com/business/live-news/wallstr...,NaN,198.0,2
2,AAPL,Apple Inc.,Google is being sued in Britain for potential ...,news_article,NaN,https://www.cnn.com/2025/04/16/tech/google-uk-...,NaN,241.0,3
3,AAPL,Apple Inc.,Nvidia was a $360 billion company at the start...,news_article,2005-04-15,https://finance.yahoo.com/news/nvidia-owns-4-a...,NaN,19.0,3
4,AAPL,Apple Inc.,The United States has cut off Huawei’s access ...,news_article,NaN,https://www.cnn.com/2020/08/17/tech/huawei-us-...,NaN,37.0,3


In [8]:
# get evidence groups

groups = df_sampled.groupby("ticker")

examples = []

for ticker, group in df_sampled.groupby("ticker"):
    
    company = group["company_name"].iloc[0]
    
    evidence_list = []
    
    for _, row in group.iterrows():
        evidence_list.append({
            "text": row["evidence_text"],
            "source_type": row["form_type"],
            "date": row["filing_date"],
            "url": row["source_url"]
        })
    
    examples.append({
        "ticker": ticker,
        "company_name": company,
        "evidence": evidence_list,
        "metadata": {
            "original_group_size": group["original_group_size"].iloc[0],
            "sampled_group_size": group["sampled_group_size"].iloc[0]
        }
    })

examples[0]

{'ticker': 'AAL',
 'company_name': 'American Airlines Group Inc.',
 'evidence': [{'text': "• Trump inauguration, Q4 earnings season will be in focus in the holiday-shortened week ahead. • With its transformative business model and clear growth trajectory, Netflix looks like a compelling buy for investors seeking quality growth. • Procter & Gamble faces operational challenges and tepid growth, making it less appealing in the current market environment. • Looking for more actionable trade ideas? Subscribe here for 50% off InvestingPro! U.S. stocks rallied on Friday ahead of the inauguration of Donald Trump, as the Dow Jones Industrial Average and the S&P 500 had their best week since the November election amid signs of easing inflation. For the week, the Dow and S&P 500 advanced 3.7% and 2.9%, respectively, while the tech-heavy Nasdaq Composite climbed 2.5%. Source: Investing.com The week ahead is expected to be another eventful one as investors continue to gauge the outlook for the econ

In [15]:
# save the examples to json 

with open('./data/examples.pkl', 'wb') as file: 
    pickle.dump(examples, file)